# Libya Land-Cover Assistant

**Click ▶ Start assistant below. Wait for the app, then follow the steps on its screen.**

Start a new map and chat in English in this notebook. You do not need a file from another notebook. No API key or payment is required by the app.

Tripoli is configured for this version. Use a Google account with access granted by the project owner. The current download is a small sample, not the whole region.


In [ ]:
# @title ▶ Start assistant
import sys, subprocess, traceback
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
from contextlib import redirect_stdout, redirect_stderr
from IPython.display import display, HTML, clear_output

SERVICE_COMMIT = "39df4d729de8f7158afdc75675060b6d3b1a85cb"
EXPECTED_VERSION = "0.7.1"

def start_assistant():
    global assistant, ui, local_parser
    loaded = sys.modules.get("libya_landcover")
    if loaded is not None and loaded.__version__ != EXPECTED_VERSION:
        display(HTML("<p>This session has an older assistant open. Save any finished work first, then choose "
            "<b>Runtime → Disconnect and delete runtime</b>. Click Start assistant again.</p>"))
        return
    if "ui" in globals():
        ui.close()
    clear_output(wait=True)
    progress = display(HTML("<h3>Opening your assistant…</h3><p>1 of 2 · Preparing the app. You do not need to do anything yet.</p>"), display_id=True)
    log_path = Path("/content/landcover-startup.log")
    try:
        if loaded is None:
            with log_path.open("w") as log:
                try:
                    torch_version = tuple(int(n) for n in version("torch").split("+")[0].split(".")[:2])
                except PackageNotFoundError:
                    torch_version = (0, 0)
                if torch_version < (2, 6):
                    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "torch>=2.6,<3",
                        "--index-url", "https://download.pytorch.org/whl/cpu"], stdout=log, stderr=log, check=True)
                subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                    "libya-landcover-assistant[local-chat] @ git+https://github.com/hamedsabzchi/libya-automated-land-cover-mapping.git@" + SERVICE_COMMIT],
                    stdout=log, stderr=log, check=True)
        import libya_landcover
        if libya_landcover.__version__ != EXPECTED_VERSION:
            raise RuntimeError("Unexpected assistant version")
        from google.colab import files, output
        from libya_landcover.assistant import Assistant, TRIPOLI
        from libya_landcover.job_store import JobStore
        from libya_landcover.local_language import LocalParser
        from libya_landcover.chat_language import GuidedParser
        from libya_landcover.chat_ui import display_chat
        output.enable_custom_widget_manager()
        progress.update(HTML("<h3>Opening your assistant…</h3><p>2 of 2 · Loading English chat. "
            "The first download can take several minutes. Please wait here.</p>"))
        try:
            if "local_parser" not in globals() or not isinstance(local_parser, LocalParser):
                with log_path.open("a") as log, redirect_stdout(log), redirect_stderr(log):
                    local_parser = LocalParser.load(cache_dir="/content/landcover-chat-model")
            parser = local_parser
        except Exception:
            with log_path.open("a") as log:
                traceback.print_exc(file=log)
            parser = GuidedParser()
        store = JobStore(Path("/content/assistant-store"))
        assistant = Assistant(store, engine_commit=SERVICE_COMMIT, profile=dict(TRIPOLI), parser=parser)

        def connect_google():
            import ee
            ee.Authenticate()
            ee.Initialize(project=assistant.profile["project"])
            try:
                ee.data.getAsset(assistant.profile["aoi_asset"])
                ee.data.getAsset(assistant.profile["training_asset"])
            except Exception:
                raise ValueError("Google signed in, but the configured study data could not be opened. "
                    "Ask the project owner to check your access to the Tripoli boundary and training data, then reconnect.") from None

        clear_output(wait=True)
        ui = display_chat(assistant, download=files.download, connect=connect_google)
    except Exception as exc:
        with log_path.open("a") as log:
            traceback.print_exc(file=log)
        progress.update(HTML("<h3>The assistant could not open</h3><p>Click Start assistant once more. "
            "If the problem continues, send the project owner this error name: <b>" + type(exc).__name__ + "</b>.</p>"))

start_assistant()
